In [7]:
import os
import sys
import csv
import glob
import pandas as pd
from statistics import mean

import pandas as pd
import os

pred_method = "openfold3"

original_directory = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/original_pdbs"## Native PDB directory
folder_path = f"openfold3/"

# For OpenFold3, get top-level directories
complex_list2 = [
    f for f in os.listdir(folder_path)
    if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith(".")
]

complex_list = [f.replace(".result", "") for f in complex_list2]
print(complex_list)

## create results folder
result_folder = pred_method + "_results"
result_path = os.path.join("/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/", result_folder)
os.makedirs(result_path, exist_ok=True)

# Ensure DockQ is in your path or installed
try:
    from DockQ.DockQ import load_PDB, run_on_all_native_interfaces
except ImportError:
    print("Error: Could not import DockQ. Please ensure the DockQ repository is in your PYTHONPATH.")
    sys.exit(1)

# --- CONFIGURATION ---
ROOT_INPUT_FOLDER = folder_path
ORIGINAL_PDB_FOLDER = original_directory

FINAL_SUMMARY_FILE = "all_dockq_scores.csv"
ERROR_LOG_FILE = "error_log.txt"
# ---------------------

def log_error(message, error_file_path):
    """Appends an error message to the log file."""
    print(f"XXX ERROR: {message}")
    with open(error_file_path, "a") as f:
        f.write(f"{message}\n")

def merge_chains(model, chains_to_merge):
    """Merges specified chains in the given model."""
    for chain in chains_to_merge[1:]:
        for res in list(model[chain]):
            res.id = (chains_to_merge[0], res.id[1], res.id[2])
            model[chains_to_merge[0]].add(res)
        model.detach_child(chain)
    model[chains_to_merge[0]].id = "".join(chains_to_merge)
    return model

def calculate_dockq(model, native, chain_map):
    """Calculates DockQ scores."""
    try:
        results, dockq_score = run_on_all_native_interfaces(model, native, chain_map=chain_map)
        return results, dockq_score
    except Exception as e:
        raise RuntimeError(f"DockQ internal calculation error: {e}")

def process_models(models, error_log_path):
    """Processes the provided models and calculates DockQ scores."""
    results_list = []
    
    for model_file, native_file in models:
        model_id = os.path.basename(model_file)
        native_id = os.path.basename(native_file)
        
        print(f"Processing model: {model_id}")
        
        try:
            model = load_PDB(model_file)
            native = load_PDB(native_file)
        except Exception as e:
            log_error(f"{model_id}: Failed to load PDB files. Error: {e}", error_log_path)
            continue

        chain_ids = list(model.child_dict.keys())
        native_chain_ids = list(native.child_dict.keys())

        try:
            # --- Logic for 3 chains (Merge first two) ---
            if len(chain_ids) == 3:
                if len(native_chain_ids) < 3:
                    raise ValueError(f"Model has 3 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")

                model_merged = merge_chains(model, chain_ids[:2])
                native_merged = merge_chains(native, native_chain_ids[:2])
                
                curr_model_chains = list(model_merged.child_dict.keys())
                curr_native_chains = list(native_merged.child_dict.keys())
                
                if len(curr_native_chains) < 2:
                    raise ValueError(f"Native structure chain merge failed. Resulting chains: {curr_native_chains}")

                chain_map_merged = {curr_native_chains[1]: curr_model_chains[1], curr_native_chains[0]: curr_model_chains[0]}
                
                results_merged, _ = calculate_dockq(model_merged, native_merged, chain_map_merged)
                
                if results_merged:
                    merged_result = results_merged[list(results_merged.keys())[0]]
                    results_list.append((
                        model_id, merged_result['DockQ'], merged_result['fnat'],
                        merged_result['iRMSD'], merged_result['LRMSD'], merged_result['F1']
                    ))

            # --- Logic for 2 chains ---
            elif len(chain_ids) == 2:
                if len(native_chain_ids) < 2:
                    raise ValueError(f"Model has 2 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")
                
                try:
                    target_native_chain_1 = native_chain_ids[0]
                    target_native_chain_2 = native_chain_ids[1]
                except IndexError:
                    raise ValueError(f"Native {native_id} missing chains at index 0 or 1.")

                chain_map = {target_native_chain_1: chain_ids[0], target_native_chain_2: chain_ids[1]}
                
                results, _ = calculate_dockq(model, native, chain_map)
                
                if results:
                    first_key = list(results.keys())[0]
                    results_list.append((
                        model_id, results[first_key]['DockQ'], results[first_key]['fnat'],
                        results[first_key]['iRMSD'], results[first_key]['LRMSD'], results[first_key]['F1']
                    ))
            
            else:
                log_error(f"{model_id}: Skipping. Model has {len(chain_ids)} chains (only 2 or 3 supported).", error_log_path)

        except KeyError as e:
            log_error(f"{model_id}: Chain ID Error. Missing chain in native or model. Details: {e}", error_log_path)
        except ValueError as e:
            log_error(f"{model_id}: Structure Mismatch. {e}", error_log_path)
        except Exception as e:
            log_error(f"{model_id}: Unexpected Error during processing. {e}", error_log_path)

    return results_list

def save_results_to_csv(results, filename):
    if not results:
        return
    print(f"Saving results to: {filename}")
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['model_id', 'DockQ', 'fnat', "iRMSD", "LRMSD", "F1"])
        for row in results:
            writer.writerow(row)

def process_folder(current_dir, original_directory, error_log_path, complex_id):
    pdb_files = glob.glob(os.path.join(current_dir, "result_sample_*_model.pdb"))
    
    if not pdb_files:
        return None

    models = []

    # Match all model files against the same native, derived from the top-level folder name
    native_candidate = os.path.join(original_directory, f"{complex_id}.pdb")
    
    if not os.path.exists(native_candidate):
        log_error(f"{complex_id}: Native file not found at {native_candidate}", error_log_path)
        return None

    for pdb_file in pdb_files:
        models.append((pdb_file, native_candidate))

    if not models:
        return None

    results = process_models(models, error_log_path)

    if results:
        csv_name = f"{complex_id}_dockq_scores.csv"
        output_path = os.path.join(current_dir, csv_name)
        save_results_to_csv(results, output_path)
        return output_path
    
    return None
    
def main():
    print(f"Starting Scan in: {ROOT_INPUT_FOLDER}")
    print(f"Using Natives from: {ORIGINAL_PDB_FOLDER}")
    
    error_log_path = os.path.join(ROOT_INPUT_FOLDER, ERROR_LOG_FILE)
    with open(error_log_path, "w") as f:
        f.write("--- DockQ Processing Error Log ---\n")

    all_csv_files = []

    for root, dirs, files in os.walk(ROOT_INPUT_FOLDER):
        if any("_sample_" in f and "_model.pdb" in f for f in files):

            rel_path = os.path.relpath(root, ROOT_INPUT_FOLDER)
            top_level_folder = rel_path.split(os.sep)[0]        
            complex_id = top_level_folder.replace("_of3", "")   
            
            print(f"\n--- Processing Folder: {root} (Complex: {complex_id}) ---")
            created_csv = process_folder(root, ORIGINAL_PDB_FOLDER, error_log_path, complex_id)
            if created_csv:
                all_csv_files.append(created_csv)

if __name__ == "__main__":
    main()

['8cyj_H_A_of3', '7wtf_H_L_A_of3', '8da1_H_L_A_of3', '7zwi_H_L_A_of3', '8hpv_H_L_A_of3', '8b8i_H_A_of3', '7wcp_H_L_A_of3', '8oxy_H_L_A_of3', '7x2l_H_A_of3', '7x8p_H_L_A_of3', '8fbw_H_L_A_of3', '7tow_H_L_A_of3', '7x2j_H_A_of3', '7anq_B_A_of3', '8g2m_H_L_A_of3', '7yds_H_L_A_of3', '7nfq_C_A_of3', '7daa_H_L_A_of3', '8hxq_H_A_of3', '8dqu_H_A_of3', '8ce4_H_A_of3', '7usv_H_A_of3', '8f6o_H_L_A_of3', '8c3l_H_A_of3', '8d36_H_L_A_of3', '7nxx_H_A_of3', '8f9f_H_L_A_of3', '8f95_H_L_A_of3', '8dgu_H_L_A_of3', '8f9t_H_L_A_of3', '7q6c_H_A_of3', '7voa_H_A_of3', '8d47_H_L_A_of3', '7zxu_H_A_of3', '7zk1_H_A_of3', '7unz_H_A_of3', '8da0_H_L_A_of3', '7uz7_H_L_A_of3', '7q6c_H_L_A_of3', '8che_H_L_A_of3', '8heb_H_L_A_of3', '8h5u_H_A_of3', '7uz8_H_L_A_of3', '7q4q_H_L_A_of3', '7unb_H_L_A_of3', '7wti_H_L_A_of3', '8i5h_H_L_A_of3', '7uvh_H_L_A_of3', '8qf5_H_A_of3', '7wpe_H_L_A_of3', '8ath_H_L_A_of3', '8ds5_H_L_A_of3', '7tcq_H_L_A_of3', '7wro_H_L_A_of3', '7xy8_H_L_A_of3', '7s11_I_M_D_of3', '8fb6_H_L_A_of3', '8oxw_H_L_A

In [12]:
import os
import glob
import pandas as pd

root_dir = "openfold3/" 
output_file = "all_dockq_scores_combined.csv"

csv_files = glob.glob(os.path.join(root_dir, "**", "*_dockq_scores.csv"), recursive=True)

print(f"Found {len(csv_files)} CSV files.")

df_list = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        df["source_folder"] = os.path.dirname(f)
        df_list.append(df)
    except pd.errors.EmptyDataError:
        print(f"Skipping empty file: {f}")

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    combined_df.to_csv(output_file, index=False)
    print(f"Saved {len(combined_df)} rows to: {output_file}")
else:
    print("No valid data found.")

Found 104 CSV files.
Saved 2600 rows to: all_dockq_scores_combined.csv


In [14]:
import pandas as pd

def classify_dockq(score):
    if 0.00 <= score < 0.23:
        return 'Incorrect'
    elif 0.23 <= score < 0.49:
        return 'Acceptable'
    elif 0.49 <= score < 0.80:
        return 'Medium'
    elif score >= 0.80:
        return 'High'
    else:
        return 'Invalid score'

def classify_capri(row):
    fnat, i_rmsd, l_rmsd = row['fnat'], row['iRMSD'], row['LRMSD']

    # High: fnat ≥ 0.5 AND (L-RMSD ≤ 1.0 OR i-RMSD ≤ 1.0)
    if fnat >= 0.5 and (l_rmsd <= 1.0 or i_rmsd <= 1.0):
        return "High"

    # Medium: fnat ≥ 0.3 AND (L-RMSD ≤ 5.0 OR i-RMSD ≤ 2.0)
    if fnat >= 0.3 and (l_rmsd <= 5.0 or i_rmsd <= 2.0):
        return "Medium"

    # Acceptable: fnat ≥ 0.1 AND (L-RMSD ≤ 10.0 OR i_RMSD ≤ 4.0)
    if fnat >= 0.1 and (l_rmsd <= 10.0 or i_rmsd <= 4.0):
        return "Acceptable"

    # Otherwise, Incorrect
    return "Incorrect"

df=pd.read_csv("all_dockq_scores_combined.csv")
print(list(df))
df['CAPRI'] = df.apply(classify_capri, axis=1)
df['CAPRI'] = df.apply(classify_capri, axis=1)
print(list(df))
df.to_csv("all_dockq_scores_combined_fin.csv",index=False)
df

['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'source_folder']
['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'source_folder', 'CAPRI']


,model_id,DockQ,fnat,iRMSD,LRMSD,F1,source_folder,CAPRI
0,result_sample_23_model.pdb,0.484149,0.564103,2.860886,5.928781,0.517647,openfold3/8cyj_H_A_of3/results,Acceptable
1,result_sample_11_model.pdb,0.357665,0.615385,3.588115,12.715603,0.448598,openfold3/8cyj_H_A_of3/results,Acceptable
2,result_sample_6_model.pdb,0.377244,0.615385,3.224851,11.883042,0.432432,openfold3/8cyj_H_A_of3/results,Acceptable
3,result_sample_20_model.pdb,0.360423,0.615385,3.558706,12.533902,0.413793,openfold3/8cyj_H_A_of3/results,Acceptable
4,result_sample_22_model.pdb,0.475264,0.589744,2.958305,6.492618,0.464646,openfold3/8cyj_H_A_of3/results,Acceptable
...,...,...,...,...,...,...,...,...
2595,result_sample_15_model.pdb,0.008596,0.000000,17.431594,62.019887,0.000000,openfold3/7z1c_H_A_of3/results,Incorrect
2596,result_sample_25_model.pdb,0.009565,0.000000,15.960293,59.593210,0.000000,openfold3/7z1c_H_A_of3/results,Incorrect
2597,result_sample_4_model.pdb,0.010356,0.000000,15.798301,56.499157,0.000000,openfold3/7z1c_H_A_of3/results,Incorrect
2598,result_sample_5_model.pdb,0.183225,0.310345,3.985658,23.549403,0.333333,openfold3/7z1c_H_A_of3/results,Acceptable


In [19]:
import os
import glob
import pandas as pd
import json
from Bio.PDB import PDBIO
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Selection import unfold_entities
import numpy as np
import sys
import argparse
import pickle
import itertools

### Interface score extraction functions

def retrieve_IFplddt(structure, chain1, chain2_lst, max_dist):
    chain_lst = list(chain1) + chain2_lst
    ifplddt = []
    contact_chain_lst = []
    for res1 in structure[0][chain1]:
        for chain2 in chain2_lst:
            count = 0
            for res2 in structure[0][chain2]:
                if res1.has_id('CA') and res2.has_id('CA'):
                   dis = abs(res1['CA']-res2['CA'])
                   if dis <= max_dist:
                      ifplddt.append(res1['CA'].get_bfactor())
                      count += 1
                elif res1.has_id('CB') and res2.has_id('CB'):
                   dis = abs(res1['CB']-res2['CB'])
                   if dis <= max_dist:
                      ifplddt.append(res1['CB'].get_bfactor())
                      count += 1
            if count > 0:
              contact_chain_lst.append(chain2)
    contact_chain_lst = sorted(list(set(contact_chain_lst)))   
    if len(ifplddt)>0:
       IF_plddt_avg = np.mean(ifplddt)
    else:
       IF_plddt_avg = 0
    return IF_plddt_avg, ifplddt, contact_chain_lst


def retrieve_IFPDEinter(structure, pdeMat, contact_lst, max_dist):
    chain_lst = [x.id for x in structure[0]]
    seqlen = [len(x) for x in structure[0]]
    ifpde_per_chain = {}
    ifpde_all = []
    
    for ch1_idx in range(len(chain_lst)):
        chain_id = chain_lst[ch1_idx]
        idx = chain_lst.index(chain_id)
        ch1_sta = sum(seqlen[:idx])
        ch1_end = ch1_sta + seqlen[idx]
        ifpde_col = []   
        
        for contact_ch in contact_lst[ch1_idx]:
            index = chain_lst.index(contact_ch)
            ch_sta = sum(seqlen[:index])
            ch_end = ch_sta + seqlen[index]
            pdeMat = np.array(pdeMat)
            remain_pdeMatrix = pdeMat[ch1_sta:ch1_end, ch_sta:ch_end]
            mat_x = -1
            
            for res1 in structure[0][chain_id]:
                mat_x += 1
                mat_y = -1
                for res2 in structure[0][contact_ch]:
                    mat_y += 1
                    if res1['CA'] - res2['CA'] <= max_dist:
                        ifpde_col.append(remain_pdeMatrix[mat_x, mat_y])
        
        if not ifpde_col:
            ifpde_per_chain[chain_id] = 0
        else:
            ifpde_per_chain[chain_id] = np.mean(ifpde_col)
            ifpde_all.extend(ifpde_col)
    
    return ifpde_per_chain, ifpde_all

def retrieve_IFPAEinter(structure, paeMat, contact_lst, max_dist):
    """Normalized inter-chain interface PAE per chain (from pDockQ2 paper)."""
    chain_lst = [x.id for x in structure[0]]
    seqlen = [len(x) for x in structure[0]]
    ifpae_avg = []
    d = 10

    for ch1_idx in range(len(chain_lst)):
        idx = chain_lst.index(chain_lst[ch1_idx])
        ch1_sta = sum(seqlen[:idx])
        ch1_end = ch1_sta + seqlen[idx]
        ifpae_col = []

        for contact_ch in contact_lst[ch1_idx]:
            index = chain_lst.index(contact_ch)
            ch_sta = sum(seqlen[:index])
            ch_end = ch_sta + seqlen[index]
            paeMat = np.array(paeMat)
            remain_paeMatrix = paeMat[ch1_sta:ch1_end, ch_sta:ch_end]

            mat_x = -1
            for res1 in structure[0][chain_lst[ch1_idx]]:
                mat_x += 1
                mat_y = -1
                for res2 in structure[0][contact_ch]:
                    mat_y += 1
                    if res1['CA'] - res2['CA'] <= max_dist:
                        ifpae_col.append(remain_paeMatrix[mat_x, mat_y])
        if not ifpae_col:
            ifpae_avg.append(0)
        else:
            norm_if_interpae = np.mean(1 / (1 + (np.array(ifpae_col) / d) ** 2))
            ifpae_avg.append(norm_if_interpae)
    return ifpae_avg
def sigmoid(x, L, x0, k, b):
    y = L / (1 + np.exp(-k * (x - x0))) + b
    return y

def calc_pmidockq(ifpae_norm, ifplddt):
    """Calculate pDockQ2 scores from normalized PAE and interface pLDDT."""
    df = pd.DataFrame()
    df['ifpae_norm'] = ifpae_norm
    df['ifplddt'] = ifplddt
    df['prot'] = df.ifpae_norm * df.ifplddt
    fitpopt = [1.31034849e+00, 8.47326239e+01, 7.47157696e-02, 5.01886443e-03]
    df['pmidockq'] = sigmoid(df.prot.values, *fitpopt)
    return df

def process_pdb_file(pdb_file, json_file, distance, file_id, chains_part=""): 
    pdbp = PDBParser(QUIET=True)
    structure = pdbp.get_structure('', pdb_file)
    chains = [chain.id for chain in structure[0]]
    remain_contact_lst = []
    plddt_lst = []
    
    for idx in range(len(chains)):
        chain_id = chains[idx]
        chain2_lst = list(set(chains) - set(chain_id))
        IF_plddt, ifplddt_vals, contact_lst = retrieve_IFplddt(structure, chain_id, chain2_lst, distance)
        plddt_lst.append(IF_plddt)
        remain_contact_lst.append(contact_lst)
    
    with open(json_file, 'r') as f:
        pde_data = json.load(f)
    
    pde_matrix = pde_data.get("pde", pde_data.get("pae", None))
    if pde_matrix is None:
        raise ValueError(f"Neither 'pde' nor 'pae' found in {json_file}")

    pde_matrix_np = np.array(pde_matrix)

    ifpde_per_chain, ifpde_all = retrieve_IFPDEinter(structure, pde_matrix_np, remain_contact_lst, distance)

    ifpae_norm = retrieve_IFPAEinter(structure, pde_matrix_np, remain_contact_lst, distance)
    res = calc_pmidockq(ifpae_norm, plddt_lst)
    
    pdb_id = os.path.basename(pdb_file).split('_')[0]
    
    result = {
        "model_id":           os.path.basename(pdb_file),
        "pdb_id":             os.path.dirname(pdb_file),
        "pdb_id_with_chains": '{0}_{1}'.format(pdb_id, chains_part) if chains_part else pdb_id,
        "chains":             "_".join(sorted(chains)),
        "ifpde_per_chain":    {k: float(v) for k, v in ifpde_per_chain.items()},
        "pde_max":            float(pde_matrix_np.max()),
        "ifpae_norm_ag":      res['ifpae_norm'].tolist()[-1],
        "ifpae_norm_avg":     float(np.mean(res['ifpae_norm'])),
        "ifplddt_ag":         res['ifplddt'].tolist()[-1],
        "ifplddt_avg":        float(np.mean(res['ifplddt'])),
        "PdockQ2 Antigen":    res['pmidockq'].tolist()[-1],
        "PdockQ2_avg":        float(np.mean(res['pmidockq'])),
    }
    
    return result

def find_matching_json(pdb_file, directory):
    pdb_file_basename = os.path.basename(pdb_file)
    json_filename = pdb_file_basename.replace("_model.pdb", "_confidences.json")
    json_path = os.path.join(directory, json_filename)
    if os.path.exists(json_path):
        return json_path
    else:
        print(f"JSON file not found: {json_path}")
        return None

def remove_existing_csvs(folder_path, pattern="*_pdockq2_scores.csv"):
    csv_files = glob.glob(os.path.join(folder_path, "**/" + pattern), recursive=True)
    if csv_files:
        print(f"\nFound {len(csv_files)} existing CSV files to remove:")
        for csv_file in csv_files:
            try:
                os.remove(csv_file)
                print(f"  ✓ Removed: {csv_file}")
            except Exception as e:
                print(f"  ✗ Error removing {csv_file}: {e}")
    else:
        print("\nNo existing CSV files found to remove.")

def run_processing(current_dir, result_output_path):
    pdb_files = glob.glob(os.path.join(current_dir, "result_sample_*_model.pdb"))
    if not pdb_files:
        return None
    results = []    
    current_file_id = "unknown_complex"
    
    for pdb_file in pdb_files:
        pdb_file_basename = os.path.basename(pdb_file)
        pdb_id = pdb_file_basename.split('_seed_')[0]
        parts = pdb_file_basename.replace("_model.pdb", "").split('_')
        chains_part = "_".join(parts[1:]) if len(parts) > 1 else ""
        json_file_name = find_matching_json(pdb_file, current_dir)
        
        if json_file_name and os.path.exists(json_file_name):
            print(f"Processing: {pdb_file_basename}")
            current_file_id = pdb_id
            try:
                result = process_pdb_file(pdb_file, json_file_name, 8, pdb_id, chains_part)
                print(f"  pDockQ2 avg: {result['PdockQ2_avg']:.4f}, Antigen: {result['PdockQ2 Antigen']:.4f}")
                print(f"  pde_max: {result['pde_max']:.3f}, ifpae_norm_avg: {result['ifpae_norm_avg']:.4f}")
                results.append(result)
            except Exception as e:
                print(f"Error processing {pdb_file}: {e}")
                continue
        else:
            print(f"JSON file for {pdb_file} not found.")
            continue

    if results:
        df = pd.DataFrame(results)
        csv_filename = f"{current_file_id}_interface_scores.csv"
        csv_file_path = os.path.join(current_dir, csv_filename)
        df.to_csv(csv_file_path, index=False)
        print(f"Data saved to {csv_file_path}")
        return csv_file_path
    else:
        print(f"No results generated for {current_dir}")
        return None

def combine_csv_files(result_path, output_file=None):
    csv_files = glob.glob(os.path.join(result_path, "**/*_interface_scores.csv"), recursive=True)
    if not csv_files:
        print("No CSV files found to combine.")
        return pd.DataFrame()
    print(f"Found {len(csv_files)} CSV files to combine")
    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df['Source_Folder'] = os.path.dirname(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        if output_file:
            output_path = os.path.join(result_path, output_file)
            combined_df.to_csv(output_path, index=False)
            print(f"Combined CSV saved to {output_path}")
        return combined_df
    else:
        return pd.DataFrame()

if __name__ == "__main__":
    
    pred_method = "openfold3"
    folder_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/"
    print("="*70)
    print("Cleaning up existing CSV files...")
    print("="*70)
    remove_existing_csvs(folder_path, "*_pdockq2_scores.csv")
    remove_existing_csvs(folder_path, "*_interface_scores.csv")
    
    complex_list2 = [
        f for f in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith(".")
    ]
    complex_list = [f.replace(".result", "") for f in complex_list2]
    print(f"\nFound {len(complex_list)} complexes to process")
    
    result_path = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/"
    os.makedirs(result_path, exist_ok=True)
    
    interface_output = f"{pred_method}_interface_scores_combined.csv"
    
    print(f"\nStarting batch processing on {len(complex_list)} complexes...")
    print("="*70)
    
    all_csv_files = []
    for root, dirs, files in os.walk(folder_path):
        if any("_sample_" in f and "_model.pdb" in f for f in files):
            print(f"\n--- Processing Folder: {root} ---")
            created_csv = run_processing(root, result_path)
            if created_csv:
                all_csv_files.append(created_csv)
    
    print("\n" + "="*70)
    print("Combining Results")
    print("="*70)
    combined_df = combine_csv_files(folder_path, interface_output)
    
    if not combined_df.empty:
        print(f"\n✓ Success! Processed {len(combined_df)} models")
        print(f"✓ Combined results saved to: {os.path.join(folder_path, interface_output)}")
    else:
        print("\n✗ No results generated.")

Cleaning up existing CSV files...

No existing CSV files found to remove.

Found 4 existing CSV files to remove:
  ✓ Removed: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/8cyj_H_A_of3/results/result_sample_10_model.pdb_interface_scores.csv
  ✓ Removed: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/7wtf_H_L_A_of3/results/result_sample_10_model.pdb_interface_scores.csv
  ✓ Removed: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/8da1_H_L_A_of3/results/result_sample_10_model.pdb_interface_scores.csv
  ✓ Removed: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/7zwi_H_L_A_of3/results/result_sample_10_model.pdb_interface_scores.csv

Found 104 complexes to process

Starting batch processing on 104 complexes...

--- Processing Folder: /media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3/8cyj_H_A_of3/results ---
Processing: result_sample_23_model.pdb
  pDockQ2 avg: 0.3189, Antigen: 0.3775
  pde_max: 5.622, ifpa

In [ ]:
import glob
import pandas as pd
import os

root_dir = "/media/emel/WD_3tb/Ab-Ag_benchmark_boltz_chai/paper/R2/openfold3"
csv_files = glob.glob(f"{root_dir}/**/metrics.csv", recursive=True)
print(f"Found {len(csv_files)} metric.csv files")

df_list = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_folder"] = os.path.dirname(f)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_csv(f"{root_dir}/combined_metrics.csv", index=False)